# Ounass Product Recommendation Model
**Kethan Pabbi — Growth Case Study Q6**

Hybrid content-based + collaborative filtering recommendation system built on session-level click data.

### Approach
- **Content-Based**: TF-IDF on product name + designer, one-hot encoding on productclass, class, gender
- **Collaborative**: Item-item similarity via session co-click matrix
- **Hybrid**: Weighted blend (alpha controls content vs collaborative weight)
- **Diversity**: Maximal Marginal Relevance (MMR) to avoid near-duplicate recommendations
- **Popularity**: Soft log-normalised popularity boost (5%) to surface relevant trending items

### Dataset
- 6.4M click events | 169K sessions | 84K unique products | January 2025

---
## 1. Setup & Imports

In [1]:
# Install dependencies if needed
%pip install pandas pyarrow scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, normalize
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

All imports successful.


---
## 2. Load Data

In [4]:
# Update path if parquet file is in a different location
df = pd.read_parquet('reco_model_data.parquet')

print(f'Loaded: {df.shape[0]:,} rows')
print(f'Unique products:  {df["product_id"].nunique():,}')
print(f'Unique sessions:  {df["session_id"].nunique():,}')
print(f'Date range:       {df["dt"].min()} to {df["dt"].max()}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(5)

Loaded: 6,477,142 rows
Unique products:  112,145
Unique sessions:  169,800
Date range:       2025-01-01 to 2025-01-31

Columns: ['dt', 'session_id', 'event_name', 'product_id', 'image_url', 'productclass', 'class', 'subclass', 'designer', 'name', 'gender']


,dt,session_id,event_name,product_id,image_url,productclass,class,subclass,designer,name,gender
0,2025-01-23,534295,click,2368,https://ounass.com//2/1/212588355_nocolor_in.j...,Beauty,"Creams, Moisturisers And Serums",Eye Treatments,Sensai,Cellular Performance Extra Intensive 10 Minute...,Unisex
1,2025-01-09,684505,click,2368,https://ounass.com//2/1/212588355_nocolor_in.j...,Beauty,"Creams, Moisturisers And Serums",Eye Treatments,Sensai,Cellular Performance Extra Intensive 10 Minute...,Unisex
2,2025-01-20,314905,click,2369,https://ounass.com//2/1/212588361_nocolor_in.j...,Beauty,Accessories And Tools,Face Brushes,Sensai,Cheek Brush,Womens
3,2025-01-26,141190,click,2369,https://ounass.com//2/1/212588361_nocolor_in.j...,Beauty,Accessories And Tools,Face Brushes,Sensai,Cheek Brush,Womens
4,2025-01-30,459680,click,2369,https://ounass.com//2/1/212588361_nocolor_in.j...,Beauty,Accessories And Tools,Face Brushes,Sensai,Cheek Brush,Womens


---
## 3. Build Product Catalogue
Deduplicate to one row per product. We use name + designer as the dedup key to remove colour/size variants that are near-identical.

In [5]:
# Deduplicate: same name + designer = same product, keep most recent
products = (
    df.sort_values('dt', ascending=False)
    .drop_duplicates(subset=['name', 'designer'])
    .drop_duplicates(subset='product_id')
    [['product_id', 'name', 'designer', 'productclass', 'class', 'subclass', 'gender', 'image_url']]
    .reset_index(drop=True)
)

# Clean text fields
for col in ['name', 'designer', 'productclass', 'class', 'subclass', 'gender']:
    products[col] = products[col].fillna('unknown').str.strip()

# Add click counts as popularity signal
click_counts = df[df['product_id'].isin(products['product_id'])].groupby('product_id').size()
products['click_count'] = products['product_id'].map(click_counts).fillna(0)
products['popularity_score'] = np.log1p(products['click_count'])
products['popularity_score'] = products['popularity_score'] / products['popularity_score'].max()

print(f'Product catalogue: {len(products):,} unique products')
print(f'\nProductclass distribution:')
print(products['productclass'].value_counts())
print(f'\nTop 10 designers by product count:')
print(products['designer'].value_counts().head(10))

Product catalogue: 84,852 unique products

Productclass distribution:
productclass
Clothing       42079
Beauty         13312
Accessories    10320
Shoes           7683
Jewellery       5764
Bags            4759
Home             931
unknown            4
Name: count, dtype: int64

Top 10 designers by product count:
designer
Dolce & Gabbana       1979
Gucci                 1360
Kith                  1195
Emporio Armani        1161
SAINT LAURENT         1071
Balenciaga            1027
Burberry               910
Prada                  793
Valentino Garavani     785
NASS                   784
Name: count, dtype: int64


---
## 4. Content-Based Feature Engineering
Two feature sets combined:
- **TF-IDF** on `designer + name + subclass` (captures brand identity and item specifics)
- **One-hot encoding** on `productclass`, `class`, `gender` (captures category structure)

Weighted: 70% TF-IDF, 30% categorical

In [6]:
# TF-IDF on text features
products['text_features'] = (
    products['designer'] + ' ' +
    products['name'] + ' ' +
    products['subclass']
)

tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(products['text_features'])
print(f'TF-IDF matrix:      {tfidf_matrix.shape}')

# One-hot encode categorical attributes
cat_features = ['productclass', 'class', 'gender']
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
cat_matrix = ohe.fit_transform(products[cat_features])
print(f'Categorical matrix: {cat_matrix.shape}')

# Combine with weights
content_matrix = hstack([tfidf_matrix * 0.7, cat_matrix * 0.3])
print(f'Combined matrix:    {content_matrix.shape}')

TF-IDF matrix:      (84852, 3000)
Categorical matrix: (84852, 153)
Combined matrix:    (84852, 3153)


---
## 5. Collaborative Filtering — Session Co-click Matrix
Products clicked together in the same session are similar. We build a sparse session-product matrix and compute item-item similarity via dot product.

In [7]:
# Build session-product co-click matrix
session_product = (
    df.groupby(['session_id', 'product_id'])
    .size()
    .reset_index(name='clicks')
)

# Keep only sessions with 2+ distinct products (meaningful co-click signal)
session_counts = session_product.groupby('session_id')['product_id'].nunique()
valid_sessions = session_counts[session_counts >= 2].index
session_product_filtered = session_product[session_product['session_id'].isin(valid_sessions)]

print(f'Sessions with 2+ products: {len(valid_sessions):,}')
print(f'Co-click events:           {len(session_product_filtered):,}')

# Build product index map
pid_to_idx = {pid: idx for idx, pid in enumerate(products['product_id'])}
products['idx'] = range(len(products))

pid_set = set(products['product_id'].tolist())
spf = session_product_filtered[session_product_filtered['product_id'].isin(pid_set)].copy()
spf['pid_idx'] = spf['product_id'].map(pid_to_idx)
spf = spf.dropna(subset=['pid_idx'])
spf['pid_idx'] = spf['pid_idx'].astype(int)

sessions = spf['session_id'].unique()
sess_to_idx = {s: i for i, s in enumerate(sessions)}
spf['sess_idx'] = spf['session_id'].map(sess_to_idx)

n_sessions = len(sessions)
n_products = len(products)

# Sparse session x product matrix
session_matrix = csr_matrix(
    (np.ones(len(spf)), (spf['sess_idx'], spf['pid_idx'])),
    shape=(n_sessions, n_products)
)

# Normalise for cosine similarity (product x session)
product_session_matrix = session_matrix.T
collab_matrix_norm = normalize(product_session_matrix, norm='l2')

print(f'\nSession-product matrix: {session_matrix.shape}')
print(f'Density: {session_matrix.nnz / (n_sessions * n_products):.4%}')
print(f'Products with session data:    {(collab_matrix_norm.sum(axis=1) > 0).sum():,}')
print(f'Products without session data: {(collab_matrix_norm.sum(axis=1) == 0).sum():,}')

Sessions with 2+ products: 145,687
Co-click events:           4,552,234

Session-product matrix: (143506, 84852)
Density: 0.0275%
Products with session data:    84,803
Products without session data: 49


---
## 6. Hybrid Recommendation Function

### Algorithm: Hybrid MMR (Maximal Marginal Relevance)

**Step 1** — Compute hybrid similarity: `alpha * content_sim + (1-alpha) * collab_sim`  
**Step 2** — Apply 5% popularity boost (log-normalised click counts)  
**Step 3** — Enforce gender consistency (penalise wrong-gender products)  
**Step 4** — Apply category boost (same productclass gets +10%)  
**Step 5** — MMR iterative selection from top-200 candidates to balance relevance vs diversity  

**Key parameters:**
- `alpha=0.6` — 60% content / 40% collaborative (default for returning users)
- `alpha=1.0` — pure content (cold start / new users)
- `diversity_penalty=0.4` — moderate MMR diversity

In [8]:
def get_recommendations(product_id, n=5, alpha=0.6, diversity_penalty=0.4,
                        candidate_pool_size=200, verbose=True):
    """
    Hybrid content + collaborative recommender with MMR diversity.
    
    Parameters:
        product_id (int): Input product to find recommendations for
        n (int): Number of recommendations to return
        alpha (float): Content vs collaborative weight. 
                       1.0 = pure content (cold start)
                       0.0 = pure collaborative
                       0.6 = default hybrid
        diversity_penalty (float): MMR diversity strength (0=none, 1=max)
        candidate_pool_size (int): How many candidates to consider before MMR
        verbose (bool): Print formatted output
    
    Returns:
        pd.DataFrame: Top-N recommendations with scores
    """
    if product_id not in pid_to_idx:
        print(f'Product {product_id} not found in catalogue')
        return None
    
    idx = pid_to_idx[product_id]
    product_info = products.iloc[idx]
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"INPUT: {product_info['designer']} — {product_info['name']}")
        print(f"       {product_info['productclass']} > {product_info['class']} > {product_info['subclass']} | {product_info['gender']}")
        print(f"       Clicks: {int(product_info['click_count']):,}")
        print(f"{'='*70}")
    
    # Step 1: Similarity scores
    content_sim = cosine_similarity(content_matrix[idx], content_matrix).flatten()
    
    collab_row = collab_matrix_norm[idx]
    has_collab = collab_row.nnz > 0
    
    if has_collab:
        collab_sim = (collab_matrix_norm @ collab_row.T).toarray().flatten()
        base_sim = alpha * content_sim + (1 - alpha) * collab_sim
        mode = f'{int(alpha*100)}% content / {int((1-alpha)*100)}% collaborative'
    else:
        base_sim = content_sim.copy()
        mode = 'content-based only (cold start — no session data)'
    
    # Step 2: Popularity blend (5%)
    pop_scores = products['popularity_score'].values
    final_sim = 0.95 * base_sim + 0.05 * pop_scores
    
    # Step 3: Gender consistency
    input_gender = product_info['gender']
    if input_gender not in ['Unisex', 'unknown']:
        wrong_gender_mask = ~products['gender'].isin([input_gender, 'Unisex', 'unknown'])
        final_sim[wrong_gender_mask.values] *= 0.3
    
    # Step 4: Category boost
    same_class_mask = (products['productclass'] == product_info['productclass']).values
    final_sim[same_class_mask] *= 1.1
    
    # Exclude self
    final_sim[idx] = -1
    
    # Step 5: MMR selection
    candidate_indices = np.argsort(final_sim)[::-1][:candidate_pool_size]
    pool = list(candidate_indices)
    pool_scores = {i: final_sim[i] for i in pool}
    
    selected = []
    selected_indices = []
    
    for _ in range(n):
        if len(pool) == 0:
            break
        
        if len(selected_indices) == 0:
            best_idx = max(pool, key=lambda i: pool_scores[i])
        else:
            selected_matrix = content_matrix[selected_indices]
            best_idx = None
            best_mmr = -999
            for candidate_idx in pool:
                relevance = pool_scores[candidate_idx]
                sim_to_selected = cosine_similarity(
                    content_matrix[candidate_idx], selected_matrix
                ).max()
                mmr = relevance - diversity_penalty * sim_to_selected
                if mmr > best_mmr:
                    best_mmr = mmr
                    best_idx = candidate_idx
        
        pool.remove(best_idx)
        selected_indices.append(best_idx)
        rec = products.iloc[best_idx]
        selected.append({
            'rank': len(selected) + 1,
            'product_id': rec['product_id'],
            'name': rec['name'],
            'designer': rec['designer'],
            'productclass': rec['productclass'],
            'class': rec['class'],
            'subclass': rec['subclass'],
            'gender': rec['gender'],
            'click_count': int(rec['click_count']),
            'similarity_score': round(float(base_sim[best_idx]), 4)
        })
    
    if verbose:
        print(f'\nTOP {n} RECOMMENDATIONS ({mode}, diversity_penalty={diversity_penalty}):\n')
        for rec in selected:
            print(f"  #{rec['rank']}  {rec['designer']:<28} {rec['name'][:55]}")
            print(f"       {rec['productclass']} > {rec['subclass']} | {rec['gender']} | {rec['click_count']:,} clicks")
            print(f"       Similarity: {rec['similarity_score']:.4f}")
            print()
    
    return pd.DataFrame(selected)

print('Recommendation function ready.')

Recommendation function ready.


---
## 7. Example Recommendations by Category
One sample product per productclass to demonstrate the model across different use cases.

In [12]:
# Sample one product per major category
sample_products = {}
for pc in ['Clothing', 'Bags', 'Shoes', 'Beauty', 'Accessories']:
    sample = products[products['productclass'] == pc].sample(1, random_state=42)
    if len(sample) > 0:
        sample_products[pc] = sample.iloc[0]['product_id']

print('Sample input products:')
for cat, pid in sample_products.items():
    p = products[products['product_id'] == pid].iloc[0]
    print(f'  {cat:<15} [{pid}] {p["designer"]} — {p["name"][:50]}')

Sample input products:
  Clothing        [120857] Mergim — Sherry Belted Maxi Dress in Silk
  Bags            [121524] Louis Vuitton Pre-Loved — 2000s Papillon 26 Shoulder Bag in Coated Monogram 
  Shoes           [110826] Michael Kors — Mandy Signature Logo Sandals in Coated-canvas
  Beauty          [57396] SkinCeuticals — 1.0% Retinol Anti Aging Night Cream, 30ml
  Accessories     [55623] CZ by Kenneth Jay Lane — Cushion & Marquise-cut CZ Drop Earrings in Rhodium


In [13]:
# Run recommendations for each category
all_results = {}
for cat, pid in sample_products.items():
    print(f'\n{"#"*70}')
    print(f'CATEGORY: {cat}')
    results = get_recommendations(pid, n=5, alpha=0.6, diversity_penalty=0.4)
    all_results[cat] = results


######################################################################
CATEGORY: Clothing

INPUT: Mergim — Sherry Belted Maxi Dress in Silk
       Clothing > Dresses > Short Sleeve | Womens
       Clicks: 34

TOP 5 RECOMMENDATIONS (60% content / 40% collaborative, diversity_penalty=0.4):

  #1  Mergim                       Lorna Belted Midi Dress in Silk
       Clothing > Short Sleeve | Womens | 54 clicks
       Similarity: 0.5725

  #2  Sruti Dalmia                 Hyacinth Belted Maxi Dress
       Clothing > Long Sleeve | Womens | 73 clicks
       Similarity: 0.4803

  #3  Harithand                    Cattleya Maxi Dress in Silk-charmeuse
       Clothing > Short Sleeve | Womens | 30 clicks
       Similarity: 0.4828

  #4  NOOM                         Prime Belted Maxi Dress in Silk
       Clothing > Long Sleeve | Womens | 69 clicks
       Similarity: 0.5224

  #5  Greek Archaic Kori           Embroidered Belted Maxi Dress in Linen
       Clothing > Short Sleeve | Womens | 28 clicks


---
## 8. Model Evaluation
Three offline metrics:
- **Coverage**: % of catalogue that appears in recommendations (higher = less concentrated)
- **Intra-list Diversity**: How different the 5 recommendations are from each other (higher = more varied)
- **Personalisation**: How unique recommendation lists are across different input products (higher = more personalised)

In [14]:
def evaluate_model(n_samples=200, n_recs=5, alpha=0.6, diversity_penalty=0.4):
    sample_pids = products['product_id'].sample(n_samples, random_state=42).tolist()
    
    all_rec_products = set()
    diversity_scores = []
    rec_sets = []
    
    for pid in sample_pids:
        recs = get_recommendations(pid, n=n_recs, alpha=alpha,
                                   diversity_penalty=diversity_penalty, verbose=False)
        if recs is not None and len(recs) > 0:
            rec_pids = recs['product_id'].tolist()
            all_rec_products.update(rec_pids)
            rec_sets.append(set(rec_pids))
            
            indices = [pid_to_idx[p] for p in rec_pids if p in pid_to_idx]
            if len(indices) > 1:
                sim_matrix = cosine_similarity(content_matrix[indices])
                np.fill_diagonal(sim_matrix, 0)
                avg_sim = sim_matrix.sum() / (len(indices) * (len(indices) - 1))
                diversity_scores.append(1 - avg_sim)
    
    personalisation_scores = []
    for i in range(min(50, len(rec_sets))):
        for j in range(i + 1, min(50, len(rec_sets))):
            overlap = len(rec_sets[i] & rec_sets[j]) / n_recs
            personalisation_scores.append(1 - overlap)
    
    coverage = len(all_rec_products) / len(products)
    avg_diversity = np.mean(diversity_scores)
    avg_personalisation = np.mean(personalisation_scores)
    
    print(f"\n{'='*52}")
    print(f' MODEL EVALUATION  (n={n_samples} samples, top-{n_recs})')
    print(f"{'='*52}")
    print(f'  Catalogue Coverage:    {coverage:.1%}  ({len(all_rec_products):,} unique products)')
    print(f'  Intra-list Diversity:  {avg_diversity:.3f}  (1.0 = max variety)')
    print(f'  Personalisation:       {avg_personalisation:.3f}  (1.0 = fully unique lists)')
    print(f"{'='*52}")
    
    return {
        'coverage': coverage,
        'diversity': avg_diversity,
        'personalisation': avg_personalisation
    }

metrics = evaluate_model()


 MODEL EVALUATION  (n=200 samples, top-5)
  Catalogue Coverage:    1.2%  (990 unique products)
  Intra-list Diversity:  0.349  (1.0 = max variety)
  Personalisation:       1.000  (1.0 = fully unique lists)


---
## 9. Cold Start vs Returning User Comparison
Demonstrates how the model adapts based on user context:
- **New user** (`alpha=1.0`): pure content — works immediately with zero session history
- **Returning user** (`alpha=0.6`): hybrid — collaborative signal enriches recommendations

In [15]:
# Pick a popular Bags product for demonstration
test_pid = (
    products[products['productclass'] == 'Bags']
    .sort_values('click_count', ascending=False)
    .iloc[5]['product_id']  # 6th most clicked bag (not top to show realistic case)
)

print('COLD START — New user (alpha=1.0, pure content-based)')
print('='*70)
cold = get_recommendations(test_pid, n=5, alpha=1.0, diversity_penalty=0.4)

print('\nRETURNING USER — Session history available (alpha=0.6, hybrid)')
print('='*70)
returning = get_recommendations(test_pid, n=5, alpha=0.6, diversity_penalty=0.4)

COLD START — New user (alpha=1.0, pure content-based)

INPUT: Gucci — GG Emblem Mini Bucket Bag in Monogram Canvas
       Bags > Shoulder Bags > Medium | Womens
       Clicks: 1,113

TOP 5 RECOMMENDATIONS (100% content / 0% collaborative, diversity_penalty=0.4):

  #1  Gucci                        GG Emblem Tote Bag in Monogram Canvas
       Bags > Medium | Womens | 153 clicks
       Similarity: 0.8794

  #2  Gucci                        GG Emblem Mini Bucket Bag in Leather
       Bags > Medium | Womens | 175 clicks
       Similarity: 0.8266

  #3  Gucci                        GG Emblem Small Shoulder Bag in Monogram Canvas
       Bags > Medium | Womens | 264 clicks
       Similarity: 0.8746

  #4  Gucci                        GG Emblem Large Shoulder Bag in Monogram Canvas
       Bags > Medium | Womens | 262 clicks
       Similarity: 0.8671

  #5  Gucci                        GG Emblem Medium Tote Bag in Monogram Canvas
       Bags > Medium | Womens | 301 clicks
       Similarity: 0.8

In [16]:
# Side-by-side comparison
print(f"\nSIDE-BY-SIDE COMPARISON")
print(f"{'Cold Start (content only)':<45} {'Returning User (hybrid)':<45}")
print('-' * 90)
for i in range(5):
    cs_name = cold.iloc[i]['name'][:43] if cold is not None else 'N/A'
    ru_name = returning.iloc[i]['name'][:43] if returning is not None else 'N/A'
    print(f'{cs_name:<45} {ru_name:<45}')


SIDE-BY-SIDE COMPARISON
Cold Start (content only)                     Returning User (hybrid)                      
------------------------------------------------------------------------------------------
GG Emblem Tote Bag in Monogram Canvas         GG Emblem Small Shoulder Bag in Monogram Ca  
GG Emblem Mini Bucket Bag in Leather          GG Emblem Mini Bucket Bag in Leather         
GG Emblem Small Shoulder Bag in Monogram Ca   GG Emblem Medium Tote Bag in Monogram Canva  
GG Emblem Large Shoulder Bag in Monogram Ca   GG Emblem Large Shoulder Bag in Monogram Ca  
GG Emblem Medium Tote Bag in Monogram Canva   GG Emblem Small Tote Bag in Monogram Canvas  


---
## 10. Diversity Penalty Comparison
Shows how the MMR diversity parameter affects recommendation spread.

In [17]:
test_pid2 = (
    products[products['productclass'] == 'Clothing']
    .sort_values('click_count', ascending=False)
    .iloc[3]['product_id']
)
p = products[products['product_id'] == test_pid2].iloc[0]
print(f'Input: {p["designer"]} — {p["name"]}\n')

for penalty in [0.0, 0.3, 0.6]:
    print(f'\n── diversity_penalty={penalty} ──')
    recs = get_recommendations(test_pid2, n=5, alpha=0.6,
                               diversity_penalty=penalty, verbose=False)
    for _, r in recs.iterrows():
        print(f"  #{int(r['rank'])}  {r['designer']:<25} {r['name'][:45]:<45} [{r['subclass']}]")

Input: Solace London — Rumi Halterneck Maxi Dress in Twill & Crepe-knit


── diversity_penalty=0.0 ──
  #1  Solace London             Filippa Maxi Dress in Twill & Crepe-knit      [Off-Shoulder]
  #2  Solace London             Maeve Maxi Dress in Twill & Crepe-knit        [Strapless]
  #3  Solace London             Flor Maxi Dress                               [Sleeveless]
  #4  Solace London             Kinsley Maxi Dress in Twill                   [Strapless]
  #5  Solace London             Kinsley Maxi Dress in Twill                   [Strapless]

── diversity_penalty=0.3 ──
  #1  Solace London             Filippa Maxi Dress in Twill & Crepe-knit      [Off-Shoulder]
  #2  Solace London             Flor Maxi Dress                               [Sleeveless]
  #3  Amri                      Halterneck Maxi Dress in Lurex-knit           [Sleeveless]
  #4  Solace London             Ophelia Halterneck Gown                       [Sleeveless]
  #5  Solace London             Kaila Cape Maxi D

---
## 11. Custom Query
Run recommendations for any product ID from the catalogue.

In [21]:
# Browse available products
print('Sample products to try:')
products[['product_id', 'designer', 'name', 'productclass', 'click_count']].sample(10, random_state=1).to_string(index=False)

Sample products to try:


' product_id            designer                                                    name productclass  click_count\n      26810                Kith                           Graham Polo T-shirt in Cotton     Clothing            9\n      10914 Christian Louboutin                         Adoxa 70 Ankle Boots in Leather        Shoes           60\n      54487               LOEWE                    Anagram Sweater Vest in Mohair-blend     Clothing           12\n      84003                Slip                                Virgo Sleep Mask in Silk       Beauty            8\n     113106              SANDRO               Yza Pocket Clutch Bag in Metallic Leather         Bags           12\n      29019             Marzook         Pill Bag in Swarovski Crystal-embellished Brass         Bags            1\n      60709        Crystal Haze Nostalgia Bear Single Earring in 18kt Gold-plated Brass  Accessories           43\n     111506          Dastaangoi                           Mini Bedouin Oud Cand

In [22]:
# ── Change product_id below to any ID from the catalogue ──
custom_product_id = products['product_id'].sample(1, random_state=99).iloc[0]

get_recommendations(
    product_id=custom_product_id,
    n=5,
    alpha=0.6,           # 0.6 = hybrid, 1.0 = content only
    diversity_penalty=0.4 # 0.0 = no diversity, 0.6 = high diversity
)


INPUT: Palm Angels — Bear Shorts in Terry Cotton
       Clothing > Shorts > Sweatshorts | Kids
       Clicks: 18

TOP 5 RECOMMENDATIONS (60% content / 40% collaborative, diversity_penalty=0.4):

  #1  Palm Angels                  Overlogo Print Shorts in Cotton
       Clothing > Sweatshorts | Kids | 8 clicks
       Similarity: 0.4886

  #2  Palm Angels                  Bear Appliqué Polo Shirt in Cotton Terry
       Clothing > Short Sleeve | Kids | 25 clicks
       Similarity: 0.4248

  #3  Burberry                     Drawcord Shorts in Terry Cloth
       Clothing > Sweatshorts | Kids | 12 clicks
       Similarity: 0.3943

  #4  Palm Angels                  Colour-block Shorts in Cotton
       Clothing > Sweatshorts | Kids | 16 clicks
       Similarity: 0.4626

  #5  Palm Angels                  Bear Graphic Hoodie in cotton
       Clothing > Hoodies | Kids | 27 clicks
       Similarity: 0.3517



,rank,product_id,name,designer,productclass,class,subclass,gender,click_count,similarity_score
0,1,109871,Overlogo Print Shorts in Cotton,Palm Angels,Clothing,Shorts,Sweatshorts,Kids,8,0.4886
1,2,109868,Bear Appliqué Polo Shirt in Cotton Terry,Palm Angels,Clothing,Polos,Short Sleeve,Kids,25,0.4248
2,3,38691,Drawcord Shorts in Terry Cloth,Burberry,Clothing,Shorts,Sweatshorts,Kids,12,0.3943
3,4,79294,Colour-block Shorts in Cotton,Palm Angels,Clothing,Shorts,Sweatshorts,Kids,16,0.4626
4,5,79295,Bear Graphic Hoodie in cotton,Palm Angels,Clothing,Sweatshirts,Hoodies,Kids,27,0.3517
